# Grounding and provenance

By default nfield returns values. Turn on grounding and it also scores how well the document
supports each value, and reports a hallucination rate. Turn on provenance and it records the
exact character span each value came from. Both are opt-in, and neither one changes the values
you get back.

## Setup

```bash
pip install "nfield[groq]"
export GROQ_API_KEY="gsk_..."
```

In [1]:
import os

from nfield import ExtractionConfig, nfield

assert os.environ.get("GROQ_API_KEY"), "set GROQ_API_KEY first"

MODEL = "groq/llama-3.3-70b-versatile"

document = """
INVOICE #4471
Vendor: Acme Corporation
Total Due: 1284.50 USD
Date: 2026-01-15
"""

schema = {
    "type": "object",
    "properties": {
        "vendor": {"type": "string"},
        "invoice_number": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["vendor", "total"],
}

## Turn both on

Pass an `ExtractionConfig` with `ground_values` and `provenance` set.

In [2]:
result = nfield(
    document,
    schema,
    MODEL,
    config=ExtractionConfig(ground_values=True, provenance=True),
)
result.data

{'vendor': 'Acme Corporation',
 'invoice_number': '4471',
 'total': 1284.5,
 'currency': 'USD'}

## The hallucination rate

With grounding on, `metadata.hallucination_rate` is the fraction of checked values the source
did not support. Lower is better; `None` means grounding was off.

In [3]:
print("hallucination rate:", result.metadata.hallucination_rate)

hallucination rate: 0.0


## Character spans

`result.provenance` maps each value to its `[start, end)` offsets in the source text, for values
that appear verbatim. Slice the document with them to show a user exactly where a value came
from.

In [4]:
for field, span in (result.provenance or {}).items():
    start, end = span
    print(f"{field:16} {span}  ->  {document[start:end]!r}")

vendor           [23, 39]  ->  'Acme Corporation'
invoice_number   [10, 14]  ->  '4471'
total            [51, 57]  ->  '1284.5'
currency         [59, 62]  ->  'USD'


## Grounding does not delete anything

A weakly supported value is reported, never dropped. A correct value is often not word for word
in the text (a unit, a derived date, an enum choice), so nfield flags rather than removes. You
decide what to do with a low score.